In [ ]:
!pip install -U ultralytics
!pip install -q onnx onnxruntime onnxscript

In [ ]:
import os
import math
import cv2
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import onnxruntime as ort

# -------------------------------
# PATHS
# -------------------------------
VIDEO_PATH = "/kaggle/input/datasets/darrenmetoyer/stage3data/HighQualityHits/TrevorLawrence (1).mp4"
HELMET_MODEL_PATH = "/kaggle/input/datasets/darrenmetoyer/stage3data/detModel1.pt"
IMPACT_MODEL_PATH = "/kaggle/input/datasets/darrenmetoyer/stage3data/clsVMode7.pt"

OUTPUT_DIR = "/kaggle/working/nfl_inference_output"
OUTPUT_VIDEO_PATH = os.path.join(OUTPUT_DIR, "annotated_sideline.mp4")
OUTPUT_ALL_DETECTIONS_CSV = os.path.join(OUTPUT_DIR, "all_scored_detections.csv")
OUTPUT_FINAL_IMPACTS_CSV = os.path.join(OUTPUT_DIR, "final_impacts.csv")

I3D_ONNX_PATH = os.path.join(OUTPUT_DIR, "i3d_binary.onnx")
ONNX_OPSET = 13
ORT_INTRA_OP_THREADS = max(1, os.cpu_count() or 1)
ORT_INTER_OP_THREADS = 1


# -------------------------------
# SETTINGS
# -------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE_HELMET = 8     
BATCH_SIZE_I3D = 4              
YOLO_IMGSZ = 1280                 
YOLO_CONF = 0.25                  
YOLO_IOU = 0.50

N_FRAMES = 9                      
STRIDE = 2                      
CROP_SIZE = 64                
BOX_EXPANSION_RATIO = 0.22       

# Notebook thresholds
DET_THRESHOLD = 0.42
CLS_THRESHOLD = 0.48
SWITCH_FRAME = 150
DET_THRESHOLD2 = 0.40
CLS_THRESHOLD2 = 0.65
DELTA_CLS = -0.07                
DELTA_DET = -0.05                
ADJ_IOU_THRESHOLD = 0.41
ADJ_MAX_FRAME_DIST = 9
ADJ_MIN_CLUSTER_SIZE = 0
ADJ_N_TIMES = 1

# Rendering
BOX_COLOR = (0, 0, 255)
TEXT_COLOR = (255, 255, 255)
TEXT_BG = (0, 0, 0)
FONT = cv2.FONT_HERSHEY_SIMPLEX

os.makedirs(OUTPUT_DIR, exist_ok=True)
cv2.setNumThreads(0)
torch.backends.cudnn.benchmark = True

# -------------------------------
# I3D MODEL
# -------------------------------
class MaxPool3dSamePadding(nn.MaxPool3d):
    def compute_pad(self, dim, s):
        if s % self.stride[dim] == 0:
            return max(self.kernel_size[dim] - self.stride[dim], 0)
        return max(self.kernel_size[dim] - (s % self.stride[dim]), 0)

    def forward(self, x):
        (_, _, t, h, w) = x.size()
        pad_t = self.compute_pad(0, t)
        pad_h = self.compute_pad(1, h)
        pad_w = self.compute_pad(2, w)
        pad = (
            pad_w // 2, pad_w - pad_w // 2,
            pad_h // 2, pad_h - pad_h // 2,
            pad_t // 2, pad_t - pad_t // 2,
        )
        x = F.pad(x, pad)
        return super().forward(x)


class Unit3D(nn.Module):
    def __init__(
        self,
        in_channels,
        output_channels,
        kernel_shape=(1, 1, 1),
        stride=(1, 1, 1),
        activation_fn=F.relu,
        use_batch_norm=True,
        use_bias=False,
    ):
        super().__init__()
        self._kernel_shape = kernel_shape
        self._stride = stride
        self._use_batch_norm = use_batch_norm
        self._activation_fn = activation_fn
        self.conv3d = nn.Conv3d(
            in_channels=in_channels,
            out_channels=output_channels,
            kernel_size=kernel_shape,
            stride=stride,
            padding=0,
            bias=use_bias,
        )
        if use_batch_norm:
            self.bn = nn.BatchNorm3d(output_channels, eps=0.001, momentum=0.01)

    def compute_pad(self, dim, s):
        if s % self._stride[dim] == 0:
            return max(self._kernel_shape[dim] - self._stride[dim], 0)
        return max(self._kernel_shape[dim] - (s % self._stride[dim]), 0)

    def forward(self, x):
        (_, _, t, h, w) = x.size()
        pad_t = self.compute_pad(0, t)
        pad_h = self.compute_pad(1, h)
        pad_w = self.compute_pad(2, w)
        pad = (
            pad_w // 2, pad_w - pad_w // 2,
            pad_h // 2, pad_h - pad_h // 2,
            pad_t // 2, pad_t - pad_t // 2,
        )
        x = F.pad(x, pad)
        x = self.conv3d(x)
        if self._use_batch_norm:
            x = self.bn(x)
        if self._activation_fn is not None:
            x = self._activation_fn(x)
        return x


class InceptionModule(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.b0 = Unit3D(in_channels, out_channels[0], kernel_shape=[1, 1, 1])
        self.b1a = Unit3D(in_channels, out_channels[1], kernel_shape=[1, 1, 1])
        self.b1b = Unit3D(out_channels[1], out_channels[2], kernel_shape=[3, 3, 3])
        self.b2a = Unit3D(in_channels, out_channels[3], kernel_shape=[1, 1, 1])
        self.b2b = Unit3D(out_channels[3], out_channels[4], kernel_shape=[3, 3, 3])
        self.b3a = MaxPool3dSamePadding(kernel_size=[3, 3, 3], stride=(1, 1, 1))
        self.b3b = Unit3D(in_channels, out_channels[5], kernel_shape=[1, 1, 1])

    def forward(self, x):
        b0 = self.b0(x)
        b1 = self.b1b(self.b1a(x))
        b2 = self.b2b(self.b2a(x))
        b3 = self.b3b(self.b3a(x))
        return torch.cat([b0, b1, b2, b3], dim=1)


class InceptionI3d(nn.Module):
    def __init__(self, num_classes=400, in_channels=3):
        super().__init__()
        self.Conv3d_1a_7x7 = Unit3D(in_channels, 64, kernel_shape=[7, 7, 7], stride=(2, 2, 2))
        self.MaxPool3d_2a_3x3 = MaxPool3dSamePadding(kernel_size=[1, 3, 3], stride=(1, 2, 2))
        self.Conv3d_2b_1x1 = Unit3D(64, 64, kernel_shape=[1, 1, 1])
        self.Conv3d_2c_3x3 = Unit3D(64, 192, kernel_shape=[3, 3, 3])
        self.MaxPool3d_3a_3x3 = MaxPool3dSamePadding(kernel_size=[1, 3, 3], stride=(1, 2, 2))
        self.Mixed_3b = InceptionModule(192, [64, 96, 128, 16, 32, 32])
        self.Mixed_3c = InceptionModule(256, [128, 128, 192, 32, 96, 64])
        self.MaxPool3d_4a_3x3 = MaxPool3dSamePadding(kernel_size=[3, 3, 3], stride=(2, 2, 2))
        self.Mixed_4b = InceptionModule(480, [192, 96, 208, 16, 48, 64])
        self.Mixed_4c = InceptionModule(512, [160, 112, 224, 24, 64, 64])
        self.Mixed_4d = InceptionModule(512, [128, 128, 256, 24, 64, 64])
        self.Mixed_4e = InceptionModule(512, [112, 144, 288, 32, 64, 64])
        self.Mixed_4f = InceptionModule(528, [256, 160, 320, 32, 128, 128])
        self.MaxPool3d_5a_2x2 = MaxPool3dSamePadding(kernel_size=[2, 2, 2], stride=(2, 2, 2))
        self.Mixed_5b = InceptionModule(832, [256, 160, 320, 32, 128, 128])
        self.Mixed_5c = InceptionModule(832, [384, 192, 384, 48, 128, 128])
        self.avg_pool = nn.AvgPool3d(kernel_size=[2, 7, 7], stride=(1, 1, 1))
        self.dropout = nn.Dropout(0.5)
        self.logits = Unit3D(1024, num_classes, kernel_shape=[1, 1, 1], activation_fn=None, use_batch_norm=False, use_bias=True)

    def extract_features(self, x):
        x = self.Conv3d_1a_7x7(x)
        x = self.MaxPool3d_2a_3x3(x)
        x = self.Conv3d_2b_1x1(x)
        x = self.Conv3d_2c_3x3(x)
        x = self.MaxPool3d_3a_3x3(x)
        x = self.Mixed_3b(x)
        x = self.Mixed_3c(x)
        x = self.MaxPool3d_4a_3x3(x)
        x = self.Mixed_4b(x)
        x = self.Mixed_4c(x)
        x = self.Mixed_4d(x)
        x = self.Mixed_4e(x)
        x = self.Mixed_4f(x)
        x = self.MaxPool3d_5a_2x2(x)
        x = self.Mixed_5b(x)
        x = self.Mixed_5c(x)
        return x

    def forward(self, x):
        x = self.extract_features(x)
        x = self.avg_pool(x)
        x = self.dropout(x)
        x = self.logits(x)
        x = x.squeeze(3).squeeze(3)
        x = torch.mean(x, 2)
        return x, None


def build_i3d_binary():
    model = InceptionI3d(num_classes=400, in_channels=3)
    model.Conv3d_1a_7x7.conv3d.stride = (1, 1, 1)
    model.MaxPool3d_4a_3x3.stride = (1, 2, 2)
    model.MaxPool3d_5a_2x2.stride = (1, 2, 2)
    nb_ft = model.logits.conv3d.in_channels
    model.logits = nn.Linear(nb_ft, 1)

    def forward_binary(x):
        x = model.extract_features(x)
        x = F.adaptive_avg_pool3d(x, (1, 1, 1))
        x = x.flatten(1)
        x = model.dropout(x)
        x = model.logits(x)
        return x, None

    model.forward = forward_binary
    return model


def load_impact_model(weight_path, device):
    model = build_i3d_binary().to(device)
    ckpt = torch.load(weight_path, map_location="cpu")

    # common checkpoint layouts
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        state = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict):
        state = ckpt
    else:
        raise ValueError("Unsupported checkpoint format for IMPACT_MODEL_PATH")

    cleaned = {}
    for k, v in state.items():
        nk = k
        for prefix in ["module.", "model."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v

    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    print(f"I3D load done | missing={len(missing)} unexpected={len(unexpected)}")
    model.eval()
    return model


class I3DOnnxWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        logits, _ = self.model(x)
        return logits


def export_i3d_to_onnx(weight_path, onnx_path, batch_size=1):
    model = load_impact_model(weight_path, device="cpu")
    wrapper = I3DOnnxWrapper(model).eval()
    dummy = torch.randn(batch_size, 3, N_FRAMES, CROP_SIZE, CROP_SIZE, dtype=torch.float32)
    os.makedirs(os.path.dirname(onnx_path), exist_ok=True)
    torch.onnx.export(
        wrapper,
        dummy,
        onnx_path,
        input_names=["clips"],
        output_names=["logits"],
        dynamic_axes={"clips": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=ONNX_OPSET,
        do_constant_folding=True,
    )
    return onnx_path


def get_onnx_session(onnx_path):
    if not os.path.exists(onnx_path):
        print(f"Exporting I3D ONNX to: {onnx_path}")
        export_i3d_to_onnx(IMPACT_MODEL_PATH, onnx_path, batch_size=1)
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads = ORT_INTRA_OP_THREADS
    so.inter_op_num_threads = ORT_INTER_OP_THREADS
    sess = ort.InferenceSession(onnx_path, sess_options=so, providers=["CPUExecutionProvider"])
    return sess

# -------------------------------
#  HELPERS
# -------------------------------
def get_adjacent_frames(frame, max_frame, n_frames=9, stride=2):
    frames = np.arange(n_frames) * stride
    frames = frames - frames[n_frames // 2] + frame
    if frames.min() < 1:
        frames -= frames.min() - 1
    elif frames.max() > max_frame:
        frames += max_frame - frames.max()
    return frames.astype(int)


def extend_box(box_xyxy, size=64):
    x1, y1, x2, y2 = box_xyxy
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    half = size / 2.0
    return np.array([cx - half, cy - half, cx + half, cy + half], dtype=np.float32)


def adapt_box_to_shape(box_xyxy, height, width):
    x1, y1, x2, y2 = box_xyxy.copy()
    bw = x2 - x1
    bh = y2 - y1

    if x1 < 0:
        x2 -= x1
        x1 = 0
    if y1 < 0:
        y2 -= y1
        y1 = 0
    if x2 > width:
        shift = x2 - width
        x1 -= shift
        x2 = width
    if y2 > height:
        shift = y2 - height
        y1 -= shift
        y2 = height

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(width, x2)
    y2 = min(height, y2)

    # final fallback in case frame is tiny or edge case occurs
    if (x2 - x1) <= 1 or (y2 - y1) <= 1:
        cx = np.clip((x1 + x2) / 2.0, 0, width)
        cy = np.clip((y1 + y2) / 2.0, 0, height)
        half = min(width, height, CROP_SIZE) / 2.0
        x1, y1, x2, y2 = cx - half, cy - half, cx + half, cy + half
        x1 = max(0, x1); y1 = max(0, y1); x2 = min(width, x2); y2 = min(height, y2)

    return np.array([int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))], dtype=np.int32)


def normalize_clip_uint8(clip_t_h_w_c):
    x = clip_t_h_w_c.astype(np.float32) / 255.0
    x = (x - 0.5) / 0.5
    x = x.transpose(3, 0, 1, 2)  # C,T,H,W
    return x


def iou_xyxy(a, b):
    xA = max(a[0], b[0])
    yA = max(a[1], b[1])
    xB = min(a[2], b[2])
    yB = min(a[3], b[3])
    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter = inter_w * inter_h
    if inter <= 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union


def expand_boxes_inplace(df, r=0.0, frame_w=1280, frame_h=720):
    df = df.copy()
    if r <= 0 or len(df) == 0:
        return df
    df["left"] = df["left"] - df["width"] * r / 2
    df["top"] = df["top"] - df["height"] * r / 2
    df["width"] = df["width"] * (1 + r)
    df["height"] = df["height"] * (1 + r)
    df["left"] = np.clip(df["left"], 0, None)
    df["top"] = np.clip(df["top"], 0, None)
    df["width"] = np.clip(df["width"], 0, frame_w - df["left"])
    df["height"] = np.clip(df["height"], 0, frame_h - df["top"])
    right = np.round(df["left"] + df["width"], 0)
    bottom = np.round(df["top"] + df["height"], 0)
    df["left"] = np.round(df["left"], 0).astype(int)
    df["top"] = np.round(df["top"], 0).astype(int)
    df["width"] = (right - df["left"]).astype(int)
    df["height"] = (bottom - df["top"]).astype(int)
    return df


def sideline_keep(row):
    det_thr = DET_THRESHOLD if row.frame <= SWITCH_FRAME else DET_THRESHOLD2
    cls_thr = CLS_THRESHOLD if row.frame <= SWITCH_FRAME else CLS_THRESHOLD2
    det_thr = det_thr - DELTA_DET
    cls_thr = cls_thr - DELTA_CLS
    return (row.det_score > det_thr) and (row.pred_cls > cls_thr)


def adjacency_postprocess(df, iou_threshold=0.41, max_dist=9, min_cluster_size=0, n_times=1):
    if len(df) == 0:
        return df.copy()

    out = df.copy().sort_values(["video", "frame"]).reset_index(drop=True)
    for _ in range(n_times):
        kept_rows = []
        for video, g in out.groupby("video", sort=False):
            g = g.sort_values("frame").reset_index(drop=True)
            clusters = [[0]] if len(g) else []
            for i in range(1, len(g)):
                assigned = False
                for cl in reversed(clusters):
                    j = cl[-1]
                    if abs(int(g.loc[i, "frame"]) - int(g.loc[j, "frame"])) > max_dist:
                        continue
                    box_i = [g.loc[i, "left"], g.loc[i, "top"], g.loc[i, "left"] + g.loc[i, "width"], g.loc[i, "top"] + g.loc[i, "height"]]
                    box_j = [g.loc[j, "left"], g.loc[j, "top"], g.loc[j, "left"] + g.loc[j, "width"], g.loc[j, "top"] + g.loc[j, "height"]]
                    if iou_xyxy(box_i, box_j) > iou_threshold:
                        cl.append(i)
                        assigned = True
                        break
                if not assigned:
                    clusters.append([i])

            centroids = []
            for cl in clusters:
                if len(cl) < min_cluster_size:
                    continue
                centroids.append(cl[len(cl) // 2])
            if len(centroids):
                kept_rows.append(g.iloc[centroids])
        out = pd.concat(kept_rows, axis=0).reset_index(drop=True) if kept_rows else out.iloc[0:0].copy()
    return out

# -------------------------------
# CLIP DATASET FOR I3D
# -------------------------------
class HelmetClipDataset(Dataset):
    def __init__(self, detections_df, frames_bgr, adj_frame_cache):
        self.frames = frames_bgr
        self.center_frames = detections_df["frame"].to_numpy(np.int32)
        self.crop_boxes = np.stack(detections_df["crop_box"].to_list()).astype(np.int32)
        self.adj_frame_cache = adj_frame_cache

    def __len__(self):
        return len(self.center_frames)

    def __getitem__(self, idx):
        center_frame = int(self.center_frames[idx])
        adj_frames = self.adj_frame_cache[center_frame]
        x1, y1, x2, y2 = self.crop_boxes[idx]

        clip = []
        for f in adj_frames:
            img = self.frames[f - 1]
            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                crop = np.zeros((CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8)
            elif crop.shape[0] != CROP_SIZE or crop.shape[1] != CROP_SIZE:
                crop = cv2.resize(crop, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_LINEAR)
            clip.append(crop[:, :, ::-1])  # BGR -> RGB

        clip = np.stack(clip, axis=0).astype(np.float32)
        clip = clip / 255.0
        clip = (clip - 0.5) / 0.5
        clip = clip.transpose(3, 0, 1, 2)  # C,T,H,W
        return torch.from_numpy(clip), idx

# -------------------------------
# READ VIDEO
# -------------------------------
assert os.path.exists(VIDEO_PATH), f"VIDEO_PATH not found: {VIDEO_PATH}"
assert os.path.exists(HELMET_MODEL_PATH), f"HELMET_MODEL_PATH not found: {HELMET_MODEL_PATH}"
assert os.path.exists(IMPACT_MODEL_PATH), f"IMPACT_MODEL_PATH not found: {IMPACT_MODEL_PATH}"

cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

frames_bgr = []
print(f"Reading video: {VIDEO_PATH}")
for _ in tqdm(range(frame_count), desc="Read frames"):
    ret, frame = cap.read()
    if not ret:
        break
    frames_bgr.append(frame)
cap.release()
frame_count = len(frames_bgr)
print(f"Frames loaded: {frame_count} | fps={fps:.2f} | size={frame_w}x{frame_h}")

# Cache temporal indices once for faster I3D clip loading
adj_frame_cache = {
    i: get_adjacent_frames(i, frame_count, n_frames=N_FRAMES, stride=STRIDE)
    for i in range(1, frame_count + 1)
}

# -------------------------------
# YOLO HELMET DETECTION
# -------------------------------
helmet_model = YOLO(HELMET_MODEL_PATH)
helmet_rows = []
print("Running helmet detection...")
for start in tqdm(range(0, frame_count, BATCH_SIZE_HELMET), desc="YOLO batches"):
    batch = frames_bgr[start:start + BATCH_SIZE_HELMET]
    results = helmet_model.predict(
        source=batch,
        conf=YOLO_CONF,
        iou=YOLO_IOU,
        imgsz=YOLO_IMGSZ,
        device=0 if DEVICE == "cuda" else "cpu",
        verbose=False,
        stream=False,
    )
    for bi, res in enumerate(results):
        frame_idx = start + bi + 1  
        if res.boxes is None or len(res.boxes) == 0:
            continue
        xyxy = res.boxes.xyxy.detach().cpu().numpy()
        confs = res.boxes.conf.detach().cpu().numpy()
        clss = res.boxes.cls.detach().cpu().numpy() if res.boxes.cls is not None else np.zeros(len(confs))
        for box, det_score, cls_id in zip(xyxy, confs, clss):
            x1, y1, x2, y2 = box.tolist()
            x1 = max(0, min(frame_w - 1, int(round(x1))))
            y1 = max(0, min(frame_h - 1, int(round(y1))))
            x2 = max(x1 + 1, min(frame_w, int(round(x2))))
            y2 = max(y1 + 1, min(frame_h, int(round(y2))))
            helmet_rows.append({
                "video": os.path.basename(VIDEO_PATH),
                "frame": frame_idx,
                "left": x1,
                "top": y1,
                "width": x2 - x1,
                "height": y2 - y1,
                "det_score": float(det_score),
                "yolo_cls": int(cls_id),
                "view": "Sideline",
            })

helmet_df = pd.DataFrame(helmet_rows)

if len(helmet_df):
    crop_boxes = []
    for row in helmet_df.itertuples(index=False):
        box_xyxy = np.array([row.left, row.top, row.left + row.width, row.top + row.height], dtype=np.float32)
        crop_box = adapt_box_to_shape(extend_box(box_xyxy, size=CROP_SIZE), frame_h, frame_w)
        crop_boxes.append(crop_box)
    helmet_df = helmet_df.copy()
    helmet_df["crop_box"] = crop_boxes

print(f"Helmet detections kept for classification: {len(helmet_df)}")

if len(helmet_df) == 0:
    print("No helmet detections found. Writing unannotated copy of video.")
    writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_w, frame_h))
    for frame in tqdm(frames_bgr, desc="Write video"):
        writer.write(frame)
    writer.release()
    pd.DataFrame(columns=["video","frame","left","top","width","height","det_score","pred_cls","impact_score"]).to_csv(OUTPUT_ALL_DETECTIONS_CSV, index=False)
    pd.DataFrame(columns=["video","frame","left","top","width","height","det_score","pred_cls","impact_score"]).to_csv(OUTPUT_FINAL_IMPACTS_CSV, index=False)
    print("Done.")
    print("OUTPUT_VIDEO_PATH:", OUTPUT_VIDEO_PATH)
    print("OUTPUT_ALL_DETECTIONS_CSV:", OUTPUT_ALL_DETECTIONS_CSV)
    print("OUTPUT_FINAL_IMPACTS_CSV:", OUTPUT_FINAL_IMPACTS_CSV)
    raise SystemExit(0)

# -------------------------------
# I3D IMPACT CLASSIFICATION (ONNX Runtime on CPU)
# -------------------------------
clip_ds = HelmetClipDataset(helmet_df, frames_bgr, adj_frame_cache)
clip_loader = DataLoader(
    clip_ds,
    batch_size=BATCH_SIZE_I3D,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

ort_session = get_onnx_session(I3D_ONNX_PATH)
ort_input_name = ort_session.get_inputs()[0].name
pred_cls = np.zeros(len(helmet_df), dtype=np.float32)
print("Running I3D impact classification with ONNX Runtime...")
for clips, indices in tqdm(clip_loader, desc="I3D ONNX batches"):
    clips_np = clips.numpy().astype(np.float32, copy=False)
    logits = ort_session.run(None, {ort_input_name: clips_np})[0]
    probs = 1.0 / (1.0 + np.exp(-logits))
    pred_cls[indices.numpy()] = probs.reshape(-1).astype(np.float32)

helmet_df["pred_cls"] = pred_cls
helmet_df["impact_score"] = helmet_df["det_score"] * helmet_df["pred_cls"]
helmet_df.drop(columns=["crop_box"], errors="ignore").to_csv(OUTPUT_ALL_DETECTIONS_CSV, index=False)
print(f"Saved scored detections: {OUTPUT_ALL_DETECTIONS_CSV}")

# -------------------------------
# SIDELINE POSTPROCESSING
# -------------------------------
print("Applying sideline thresholds and postprocessing...")
filtered_df = helmet_df[helmet_df.apply(sideline_keep, axis=1)].copy().reset_index(drop=True)
filtered_df = filtered_df.drop(columns=["crop_box"], errors="ignore")
filtered_df = expand_boxes_inplace(filtered_df, r=BOX_EXPANSION_RATIO, frame_w=frame_w, frame_h=frame_h)
for _ in tqdm(range(ADJ_N_TIMES), desc="Adjacency postprocess"):
    filtered_df = adjacency_postprocess(
        filtered_df,
        iou_threshold=ADJ_IOU_THRESHOLD,
        max_dist=ADJ_MAX_FRAME_DIST,
        min_cluster_size=ADJ_MIN_CLUSTER_SIZE,
        n_times=1,
    )
filtered_df = filtered_df.reset_index(drop=True)
filtered_df.to_csv(OUTPUT_FINAL_IMPACTS_CSV, index=False)
print(f"Final impacts kept: {len(filtered_df)}")
print(f"Saved final impacts: {OUTPUT_FINAL_IMPACTS_CSV}")

# -------------------------------
# BUILD FRAME LOOKUP FOR DRAWING
# -------------------------------
frame_to_rows = {}
for _, row in filtered_df.iterrows():
    frame_to_rows.setdefault(int(row.frame), []).append(row)

# -------------------------------
# ANNOTATED VIDEO RENDER
# -------------------------------
print("Rendering annotated video...")
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_w, frame_h))
for frame_idx, frame in enumerate(tqdm(frames_bgr, desc="Render video"), start=1):
    draw = frame.copy()
    for row in frame_to_rows.get(frame_idx, []):
        x1 = int(row.left)
        y1 = int(row.top)
        x2 = int(row.left + row.width)
        y2 = int(row.top + row.height)
        cv2.rectangle(draw, (x1, y1), (x2, y2), BOX_COLOR, 2)
        label = f"impact det={row.det_score:.2f} cls={row.pred_cls:.2f}"
        (tw, th), baseline = cv2.getTextSize(label, FONT, 0.5, 1)
        ty1 = max(0, y1 - th - baseline - 4)
        ty2 = max(th + baseline + 4, y1)
        cv2.rectangle(draw, (x1, ty1), (x1 + tw + 6, ty2), TEXT_BG, -1)
        cv2.putText(draw, label, (x1 + 3, ty2 - baseline - 2), FONT, 0.5, TEXT_COLOR, 1, cv2.LINE_AA)
    writer.write(draw)
writer.release()

print("\nDone.")
print("VIDEO_PATH used:", VIDEO_PATH)
print("Annotated video:", OUTPUT_VIDEO_PATH)
print("All scored detections CSV:", OUTPUT_ALL_DETECTIONS_CSV)
print("Final impacts CSV:", OUTPUT_FINAL_IMPACTS_CSV)

In [ ]:
import os
import cv2
import math
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

OPENCV_VIDEO_PATH = os.path.join(OUTPUT_DIR, "concussionDisplay.mp4")
OPENCV_EVENTS_CSV = os.path.join(OUTPUT_DIR, "finalConcussionDecisions.csv")

TRACK_ONLY_SECONDS = 2.0
MOTION_CHECK_SECONDS = 2.0
ANALYSIS_FRAME_STRIDE = 2

MIN_VALID_MOTION_SAMPLES = 5

SEED_FRAME_GAP = 8
SEED_IOU_THRESH = 0.35

MIN_PRED_CLS = 0.80

UPRIGHT_BODY_WIDTH_MULT = 4.0
UPRIGHT_BODY_HEIGHT_MULT = 8.0
SIDEWAYS_BODY_WIDTH_MULT = 8.0
SIDEWAYS_BODY_HEIGHT_MULT = 4.5
BODY_UP_MULT = 0.75
SIDEWAYS_ASPECT_THRESH = 1.20

REFINE_SIDEWAYS_WIDTH_MULT = 1.30
REFINE_SIDEWAYS_HEIGHT_MULT = 1.05
REFINE_UPRIGHT_WIDTH_MULT = 1.10
REFINE_UPRIGHT_HEIGHT_MULT = 1.25

BODY_MOTION_THR = 0.035
HEAD_MOTION_THR = 0.028

BODY_STRONG_MOVE_THR = 0.080
HEAD_STRONG_MOVE_THR = 0.065

HELMET_MOTION_THR = 0.030
HELMET_STRONG_MOVE_THR = 0.055

MIN_HEAD_TRACKED_POINTS = 10
MIN_BODY_TRACKED_POINTS = 16
MIN_CONFIDENT_VISIBLE_SAMPLES = 3

BODY_MISSING_STOP_SECONDS = 1.0

ROI_HIST_BINS = 32
ROI_HIST_SIMILARITY_THR = 0.55
MIN_APPEARANCE_CHECK_SAMPLES = 2

PLAYER_MAX_CORNERS = 120
PLAYER_QUALITY = 0.02
PLAYER_MIN_DISTANCE = 7
PLAYER_BLOCK_SIZE = 7

CAMERA_MAX_CORNERS = 250
CAMERA_QUALITY = 0.01
CAMERA_MIN_DISTANCE = 7
CAMERA_MIN_FEATURES = 30
CAMERA_RANSAC_THRESH = 3.0

LK_PARAMS = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.03),
)

IMPACT_DISPLAY_SECONDS = 1.0
CONCUSSION_DISPLAY_SECONDS = 2.0

FONT = cv2.FONT_HERSHEY_SIMPLEX
HELMET_GLOW_COLOR = (0, 255, 255)
CONCUSSION_GLOW_COLOR = (0, 0, 255)
HELMET_LABEL_BG = (0, 140, 255)
CONCUSSION_LABEL_BG = (0, 0, 255)
BODY_BOX_COLOR = (0, 255, 0)
HEAD_POINT_COLOR = (255, 0, 255)
BODY_POINT_COLOR = (0, 255, 0)
HEAD_ROI_COLOR = (255, 0, 255)
TORSO_ROI_COLOR = (0, 255, 0)
BANNER_BG = (0, 0, 255)
TEXT_COLOR = (255, 255, 255)

BOX_THICKNESS = 1
GLOW_THICKNESSES = [6, 4, 2]
BODY_BOX_THICKNESS = 1
POINT_RADIUS = 2
POINT_LINE_THICKNESS = 1

def box_iou_xyxy(a, b):
    xA = max(a[0], b[0])
    yA = max(a[1], b[1])
    xB = min(a[2], b[2])
    yB = min(a[3], b[3])
    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter = inter_w * inter_h
    if inter <= 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    denom = area_a + area_b - inter
    return 0.0 if denom <= 0 else inter / denom


def clamp_roi(x1, y1, x2, y2, frame_w, frame_h):
    x1 = int(max(0, min(frame_w - 1, round(x1))))
    y1 = int(max(0, min(frame_h - 1, round(y1))))
    x2 = int(max(x1 + 1, min(frame_w, round(x2))))
    y2 = int(max(y1 + 1, min(frame_h, round(y2))))
    return [x1, y1, x2, y2]


def body_scale_from_box(box_xyxy):
    x1, y1, x2, y2 = box_xyxy
    w = max(1.0, x2 - x1)
    h = max(1.0, y2 - y1)
    return max(1.0, math.hypot(w, h))


def helmet_center(box_xyxy):
    x1, y1, x2, y2 = box_xyxy
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=np.float32)


def to_gray(frame_bgr):
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)


def roi_hist_similarity(prev_gray, curr_gray, roi, bins=32):
    x1, y1, x2, y2 = roi
    a = prev_gray[y1:y2, x1:x2]
    b = curr_gray[y1:y2, x1:x2]

    if a is None or b is None or a.size == 0 or b.size == 0:
        return 0.0

    ha = cv2.calcHist([a], [0], None, [bins], [0, 256])
    hb = cv2.calcHist([b], [0], None, [bins], [0, 256])

    cv2.normalize(ha, ha)
    cv2.normalize(hb, hb)

    return float(cv2.compareHist(ha, hb, cv2.HISTCMP_CORREL))


def dedupe_seed_impacts(df):
    if len(df) == 0:
        return df.copy()

    df = df.sort_values(["frame", "impact_score"], ascending=[True, False]).reset_index(drop=True)
    keep = []

    for row in df.itertuples(index=False):
        curr_box = [row.left, row.top, row.left + row.width, row.top + row.height]
        dup = False
        for kept in keep:
            if abs(int(row.frame) - int(kept.frame)) > SEED_FRAME_GAP:
                continue
            kept_box = [kept.left, kept.top, kept.left + kept.width, kept.top + kept.height]
            if box_iou_xyxy(curr_box, kept_box) >= SEED_IOU_THRESH:
                dup = True
                break
        if not dup:
            keep.append(row)

    return pd.DataFrame([r._asdict() for r in keep])

def create_csrt_tracker():
    if hasattr(cv2, "TrackerCSRT_create"):
        return cv2.TrackerCSRT_create()
    if hasattr(cv2, "legacy") and hasattr(cv2.legacy, "TrackerCSRT_create"):
        return cv2.legacy.TrackerCSRT_create()
    raise RuntimeError("CSRT tracker not available in this OpenCV build.")


def init_tracker_on_helmet(frame_bgr, helmet_box_xyxy):
    tracker = create_csrt_tracker()
    x1, y1, x2, y2 = helmet_box_xyxy

    x = int(round(float(x1)))
    y = int(round(float(y1)))
    w = int(round(float(x2 - x1)))
    h = int(round(float(y2 - y1)))

    w = max(1, w)
    h = max(1, h)

    fh, fw = frame_bgr.shape[:2]
    x = max(0, min(fw - 1, x))
    y = max(0, min(fh - 1, y))
    w = max(1, min(w, fw - x))
    h = max(1, min(h, fh - y))

    tracker.init(frame_bgr, (x, y, w, h))
    return tracker


def update_tracker_box(tracker, frame_bgr, fallback_box):
    ok, box = tracker.update(frame_bgr)
    if not ok:
        return False, fallback_box

    x, y, w, h = box
    x = int(round(float(x)))
    y = int(round(float(y)))
    w = max(1, int(round(float(w))))
    h = max(1, int(round(float(h))))

    tracked = clamp_roi(x, y, x + w, y + h, frame_w, frame_h)
    return True, tracked

def helmet_box_to_body_roi(helmet_box_xyxy, frame_w, frame_h):
    hx1, hy1, hx2, hy2 = helmet_box_xyxy
    w = max(1.0, hx2 - hx1)
    h = max(1.0, hy2 - hy1)
    cx = 0.5 * (hx1 + hx2)

    aspect = w / h
    if aspect > SIDEWAYS_ASPECT_THRESH:
        body_w = max(w * SIDEWAYS_BODY_WIDTH_MULT, 180)
        body_h = max(h * SIDEWAYS_BODY_HEIGHT_MULT, 140)
    else:
        body_w = max(w * UPRIGHT_BODY_WIDTH_MULT, 90)
        body_h = max(h * UPRIGHT_BODY_HEIGHT_MULT, 180)

    x1 = cx - body_w / 2.0
    x2 = cx + body_w / 2.0
    y1 = hy1 - BODY_UP_MULT * h
    y2 = y1 + body_h
    return clamp_roi(x1, y1, x2, y2, frame_w, frame_h)


def refine_body_roi(body_roi):
    x1, y1, x2, y2 = body_roi
    w = max(1.0, x2 - x1)
    h = max(1.0, y2 - y1)
    cx = 0.5 * (x1 + x2)
    cy = 0.5 * (y1 + y2)

    aspect = w / h
    if aspect > 1.15:
        rw = w * REFINE_SIDEWAYS_WIDTH_MULT
        rh = h * REFINE_SIDEWAYS_HEIGHT_MULT
    else:
        rw = w * REFINE_UPRIGHT_WIDTH_MULT
        rh = h * REFINE_UPRIGHT_HEIGHT_MULT

    return clamp_roi(cx - rw / 2.0, cy - rh / 2.0, cx + rw / 2.0, cy + rh / 2.0, frame_w, frame_h)


def body_subregions_from_body_roi(body_roi):
    x1, y1, x2, y2 = body_roi
    h = max(1, y2 - y1)

    head_y2 = y1 + int(0.28 * h)
    torso_y1 = y1 + int(0.20 * h)
    torso_y2 = y1 + int(0.72 * h)

    head_roi = [x1, y1, x2, max(y1 + 1, head_y2)]
    torso_roi = [x1, torso_y1, x2, max(torso_y1 + 1, torso_y2)]
    return head_roi, torso_roi


def surrounding_ring_roi(body_roi, expand=35):
    x1, y1, x2, y2 = body_roi
    return clamp_roi(x1 - expand, y1 - expand, x2 + expand, y2 + expand, frame_w, frame_h)

def draw_helmet_highlight(
    frame,
    left,
    top,
    width,
    height,
    label="helmet impact",
    frame_idx=None,
    glow_color=None,
    label_bg=None,
):
    x1 = int(left)
    y1 = int(top)
    x2 = int(left + width)
    y2 = int(top + height)

    if glow_color is None:
        glow_color = HELMET_GLOW_COLOR
    if label_bg is None:
        label_bg = HELMET_LABEL_BG

    pulse = 0
    if frame_idx is not None:
        pulse = int(2 * abs(math.sin(frame_idx * 0.25)))

    for t in [max(1, g + pulse) for g in GLOW_THICKNESSES]:
        cv2.rectangle(frame, (x1, y1), (x2, y2), glow_color, t, cv2.LINE_AA)

    cv2.rectangle(frame, (x1, y1), (x2, y2), glow_color, BOX_THICKNESS, cv2.LINE_AA)

    (tw, th), baseline = cv2.getTextSize(label, FONT, 0.55, 1)
    ty1 = max(0, y1 - th - baseline - 6)
    ty2 = max(th + baseline + 6, y1)
    cv2.rectangle(frame, (x1, ty1), (x1 + tw + 10, ty2), label_bg, -1)
    cv2.putText(frame, label, (x1 + 5, ty2 - baseline - 4),
                FONT, 0.55, TEXT_COLOR, 1, cv2.LINE_AA)


def annotate_banner(frame, text="KNOCKED OUT / CONCUSSION SUSPECTED"):
    (tw, th), baseline = cv2.getTextSize(text, FONT, 0.85, 2)
    x1, y1 = 20, 20
    x2, y2 = x1 + tw + 14, y1 + th + baseline + 14
    cv2.rectangle(frame, (x1, y1), (x2, y2), BANNER_BG, -1)
    cv2.putText(frame, text, (x1 + 7, y2 - baseline - 7),
                FONT, 0.85, TEXT_COLOR, 2, cv2.LINE_AA)

def get_features_in_roi(gray, roi, mask=None, max_corners=100, quality=0.02, min_distance=7, block_size=7):
    x1, y1, x2, y2 = roi
    crop = gray[y1:y2, x1:x2]
    if crop is None or crop.size == 0:
        return None

    roi_mask = None
    if mask is not None:
        roi_mask = mask[y1:y2, x1:x2]

    pts = cv2.goodFeaturesToTrack(
        crop,
        maxCorners=max_corners,
        qualityLevel=quality,
        minDistance=min_distance,
        mask=roi_mask,
        blockSize=block_size,
    )
    if pts is None:
        return None

    pts[:, 0, 0] += x1
    pts[:, 0, 1] += y1
    return pts.astype(np.float32)


def track_points_lk(prev_gray, curr_gray, pts0):
    if pts0 is None or len(pts0) == 0:
        return None, None

    pts1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, pts0, None, **LK_PARAMS)
    if pts1 is None or st is None:
        return None, None

    good0 = pts0[st.flatten() == 1].reshape(-1, 2)
    good1 = pts1[st.flatten() == 1].reshape(-1, 2)

    if len(good0) == 0:
        return None, None

    return good0, good1


def estimate_camera_motion(prev_gray, curr_gray, body_roi):
    outer = surrounding_ring_roi(body_roi, expand=35)
    bx1, by1, bx2, by2 = body_roi

    mask = np.ones(prev_gray.shape, dtype=np.uint8) * 255
    mask[by1:by2, bx1:bx2] = 0

    pts0 = get_features_in_roi(
        prev_gray,
        outer,
        mask=mask,
        max_corners=CAMERA_MAX_CORNERS,
        quality=CAMERA_QUALITY,
        min_distance=CAMERA_MIN_DISTANCE,
        block_size=7,
    )

    if pts0 is None or len(pts0) < CAMERA_MIN_FEATURES:
        return np.zeros(2, dtype=np.float32)

    good0, good1 = track_points_lk(prev_gray, curr_gray, pts0)
    if good0 is None or len(good0) < CAMERA_MIN_FEATURES:
        return np.zeros(2, dtype=np.float32)

    M, _ = cv2.estimateAffinePartial2D(
        good0, good1,
        method=cv2.RANSAC,
        ransacReprojThreshold=CAMERA_RANSAC_THRESH
    )

    if M is None:
        delta = np.median(good1 - good0, axis=0)
        return delta.astype(np.float32)

    return np.array([float(M[0, 2]), float(M[1, 2])], dtype=np.float32)


def estimate_motion_in_roi(prev_gray, curr_gray, roi, camera_vec):
    pts0 = get_features_in_roi(
        prev_gray,
        roi,
        max_corners=PLAYER_MAX_CORNERS,
        quality=PLAYER_QUALITY,
        min_distance=PLAYER_MIN_DISTANCE,
        block_size=PLAYER_BLOCK_SIZE,
    )

    if pts0 is None or len(pts0) < 6:
        return None, 0, [], []

    good0, good1 = track_points_lk(prev_gray, curr_gray, pts0)
    if good0 is None or len(good0) < 6:
        return None, 0, [], []

    raw_vecs = good1 - good0
    comp_vecs = raw_vecs - camera_vec.reshape(1, 2)
    mags = np.linalg.norm(comp_vecs, axis=1)

    prev_pts = [(int(round(p[0])), int(round(p[1]))) for p in good0[:40]]
    curr_pts = [(int(round(p[0])), int(round(p[1]))) for p in good1[:40]]

    return float(np.median(mags)), int(len(good0)), prev_pts, curr_pts


def analyze_frame_pair(prev_frame_bgr, curr_frame_bgr, body_roi):
    prev_gray = to_gray(prev_frame_bgr)
    curr_gray = to_gray(curr_frame_bgr)

    camera_vec = estimate_camera_motion(prev_gray, curr_gray, body_roi)
    head_roi, torso_roi = body_subregions_from_body_roi(body_roi)

    head_motion_px, head_pts, head_prev_pts, head_curr_pts = estimate_motion_in_roi(
        prev_gray, curr_gray, head_roi, camera_vec
    )
    body_motion_px, body_pts, body_prev_pts, body_curr_pts = estimate_motion_in_roi(
        prev_gray, curr_gray, torso_roi, camera_vec
    )

    scale = body_scale_from_box(body_roi)
    head_motion_norm = None if head_motion_px is None else float(head_motion_px / scale)
    body_motion_norm = None if body_motion_px is None else float(body_motion_px / scale)

    body_visible_confidently = (head_pts >= MIN_HEAD_TRACKED_POINTS) and (body_pts >= MIN_BODY_TRACKED_POINTS)

    return {
        "head_motion": head_motion_norm,
        "body_motion": body_motion_norm,
        "body_visible_confidently": body_visible_confidently,
        "head_prev_pts": head_prev_pts,
        "head_curr_pts": head_curr_pts,
        "body_prev_pts": body_prev_pts,
        "body_curr_pts": body_curr_pts,
        "head_roi": head_roi,
        "torso_roi": torso_roi,
    }

def track_helmet_window(start_frame_idx, end_frame_idx, initial_helmet_box):
    if start_frame_idx < 1:
        start_frame_idx = 1
    if end_frame_idx > len(frames_bgr):
        end_frame_idx = len(frames_bgr)
    if start_frame_idx > end_frame_idx:
        return {}, initial_helmet_box

    tracker = init_tracker_on_helmet(frames_bgr[start_frame_idx - 1], initial_helmet_box)
    tracked_boxes = {}
    current_helmet_box = initial_helmet_box

    for frame_idx in range(start_frame_idx, end_frame_idx + 1):
        frame = frames_bgr[frame_idx - 1]

        if frame_idx == start_frame_idx:
            tracked_helmet = initial_helmet_box
        else:
            _, tracked_helmet = update_tracker_box(tracker, frame, current_helmet_box)

        current_helmet_box = tracked_helmet
        tracked_boxes[frame_idx] = tracked_helmet

    return tracked_boxes, current_helmet_box

def analyze_motion_window(start_frame_idx, end_frame_idx, initial_helmet_box):
    if start_frame_idx < 1:
        start_frame_idx = 1
    if end_frame_idx > len(frames_bgr):
        end_frame_idx = len(frames_bgr)
    if start_frame_idx > end_frame_idx:
        return [], [], {}, initial_helmet_box, False, False, False

    tracker = init_tracker_on_helmet(frames_bgr[start_frame_idx - 1], initial_helmet_box)

    motion_records = []
    helmet_motion_records = []
    per_frame_overlays = {}
    prev_frame_idx = None
    current_helmet_box = initial_helmet_box
    prev_helmet_box = initial_helmet_box

    stopped_early_for_motion = False
    stopped_early_for_visibility = False
    confident_visible_samples = 0
    bad_appearance_samples = 0
    abandoned_for_missing_body = False

    missing_body_run = 0
    missing_body_limit = max(1, int(math.ceil(BODY_MISSING_STOP_SECONDS * fps / ANALYSIS_FRAME_STRIDE)))

    for frame_idx in range(start_frame_idx, end_frame_idx + 1, ANALYSIS_FRAME_STRIDE):
        frame = frames_bgr[frame_idx - 1]

        if frame_idx == start_frame_idx:
            tracked_helmet = initial_helmet_box
        else:
            _, tracked_helmet = update_tracker_box(tracker, frame, current_helmet_box)

        current_helmet_box = tracked_helmet
        body_roi = refine_body_roi(helmet_box_to_body_roi(current_helmet_box, frame_w, frame_h))

        per_frame_overlays[frame_idx] = {
            "helmet_box": current_helmet_box,
            "body_roi": body_roi,
            "head_roi": None,
            "torso_roi": None,
            "head_prev_pts": [],
            "head_curr_pts": [],
            "body_prev_pts": [],
            "body_curr_pts": [],
        }

        if prev_frame_idx is not None:
            prev_frame = frames_bgr[prev_frame_idx - 1]
            prev_gray = to_gray(prev_frame)
            curr_gray = to_gray(frame)

            hist_sim = roi_hist_similarity(prev_gray, curr_gray, body_roi, bins=ROI_HIST_BINS)
            if hist_sim < ROI_HIST_SIMILARITY_THR:
                bad_appearance_samples += 1
                missing_body_run += 1

                if missing_body_run >= missing_body_limit:
                    stopped_early_for_visibility = True
                    abandoned_for_missing_body = True
                    motion_records = []
                    helmet_motion_records = []
                    break

                prev_frame_idx = frame_idx
                prev_helmet_box = current_helmet_box
                continue

            result = analyze_frame_pair(prev_frame, frame, body_roi)

            per_frame_overlays[frame_idx] = {
                "helmet_box": current_helmet_box,
                "body_roi": body_roi,
                "head_roi": result["head_roi"],
                "torso_roi": result["torso_roi"],
                "head_prev_pts": result["head_prev_pts"],
                "head_curr_pts": result["head_curr_pts"],
                "body_prev_pts": result["body_prev_pts"],
                "body_curr_pts": result["body_curr_pts"],
            }

            curr_center = helmet_center(current_helmet_box)
            prev_center = helmet_center(prev_helmet_box)
            helmet_px = float(np.linalg.norm(curr_center - prev_center))
            helmet_norm = helmet_px / body_scale_from_box(body_roi)
            helmet_motion_records.append(helmet_norm)

            strong_helmet = helmet_norm > HELMET_STRONG_MOVE_THR
            if strong_helmet:
                stopped_early_for_motion = True
                break

            if not result["body_visible_confidently"]:
                missing_body_run += 1

                if missing_body_run >= missing_body_limit:
                    stopped_early_for_visibility = True
                    abandoned_for_missing_body = True
                    motion_records = []
                    helmet_motion_records = []
                    break

                prev_frame_idx = frame_idx
                prev_helmet_box = current_helmet_box
                continue

            missing_body_run = 0
            confident_visible_samples += 1

            head_motion = result["head_motion"]
            body_motion = result["body_motion"]

            if head_motion is not None or body_motion is not None:
                motion_records.append(result)

            strong_head = head_motion is not None and not np.isnan(head_motion) and head_motion > HEAD_STRONG_MOVE_THR
            strong_body = body_motion is not None and not np.isnan(body_motion) and body_motion > BODY_STRONG_MOVE_THR

            if strong_head or strong_body:
                stopped_early_for_motion = True
                break

        prev_frame_idx = frame_idx
        prev_helmet_box = current_helmet_box

    if confident_visible_samples < MIN_CONFIDENT_VISIBLE_SAMPLES:
        stopped_early_for_visibility = True
        motion_records = []

    if bad_appearance_samples >= MIN_APPEARANCE_CHECK_SAMPLES and confident_visible_samples == 0:
        stopped_early_for_visibility = True
        motion_records = []
        helmet_motion_records = []

    return (
        motion_records,
        helmet_motion_records,
        per_frame_overlays,
        current_helmet_box,
        stopped_early_for_motion,
        stopped_early_for_visibility,
        abandoned_for_missing_body,
    )

def evaluate_window(motion_list, helmet_motion_list):
    head_vals = [m["head_motion"] for m in motion_list if m["head_motion"] is not None and not np.isnan(m["head_motion"])]
    body_vals = [m["body_motion"] for m in motion_list if m["body_motion"] is not None and not np.isnan(m["body_motion"])]
    helmet_vals = [v for v in helmet_motion_list if v is not None and not np.isnan(v)]

    enough_data = (
        len(head_vals) >= MIN_VALID_MOTION_SAMPLES
        and len(body_vals) >= MIN_VALID_MOTION_SAMPLES
        and len(helmet_vals) >= MIN_VALID_MOTION_SAMPLES
    )

    if not enough_data:
        return True, False, {
            "head_med": np.nan,
            "body_med": np.nan,
            "helmet_med": np.nan,
            "head_max": np.nan,
            "body_max": np.nan,
            "helmet_max": np.nan,
            "enough_data": False,
        }

    head_med = float(np.median(head_vals))
    body_med = float(np.median(body_vals))
    helmet_med = float(np.median(helmet_vals))

    head_max = float(np.max(head_vals))
    body_max = float(np.max(body_vals))
    helmet_max = float(np.max(helmet_vals))

    head_moving = head_med > HEAD_MOTION_THR
    body_moving = body_med > BODY_MOTION_THR
    helmet_moving = helmet_med > HELMET_MOTION_THR

    strong_head_movement = head_max > HEAD_STRONG_MOVE_THR
    strong_body_movement = body_max > BODY_STRONG_MOVE_THR
    strong_helmet_movement = helmet_max > HELMET_STRONG_MOVE_THR

    purposeful_movement = (
        head_moving
        or body_moving
        or helmet_moving
        or strong_head_movement
        or strong_body_movement
        or strong_helmet_movement
    )

    motionless = (
        head_med <= HEAD_MOTION_THR
        and body_med <= BODY_MOTION_THR
        and helmet_med <= HELMET_MOTION_THR
        and (not purposeful_movement)
    )

    return purposeful_movement, motionless, {
        "head_med": head_med,
        "body_med": body_med,
        "helmet_med": helmet_med,
        "head_max": head_max,
        "body_max": body_max,
        "helmet_max": helmet_max,
        "enough_data": True,
    }

if len(filtered_df) == 0:
    print("No impacts in filtered_df, so no OpenCV knockout/concussion pass was run.")
else:
    seed_df = dedupe_seed_impacts(filtered_df)
    print(f"Seed impacts to analyze with OpenCV tracking: {len(seed_df)}")

    track_only_frames = max(1, int(math.ceil(TRACK_ONLY_SECONDS * fps)))
    motion_check_frames = max(1, int(math.ceil(MOTION_CHECK_SECONDS * fps)))
    impact_display_frames = max(1, int(math.ceil(IMPACT_DISPLAY_SECONDS * fps)))
    concussion_display_frames = max(1, int(math.ceil(CONCUSSION_DISPLAY_SECONDS * fps)))

    event_rows = []
    impact_frame_overlays = {}
    concussion_frame_overlays = {}

    for event_id, row in enumerate(
        tqdm(seed_df.itertuples(index=False), total=len(seed_df), desc="Analyze impacts with OpenCV"),
        start=1
    ):
        if float(row.pred_cls) < MIN_PRED_CLS:
            continue

        impact_frame = int(row.frame)
        helmet_box_xyxy = [
            int(row.left),
            int(row.top),
            int(row.left + row.width),
            int(row.top + row.height),
        ]

        impact_display_end = min(len(frames_bgr), impact_frame + impact_display_frames - 1)
        for fidx in range(impact_frame, impact_display_end + 1):
            impact_frame_overlays.setdefault(fidx, []).append({
                "event_id": event_id,
                "helmet_box": helmet_box_xyxy,
            })

        track_start = impact_frame
        track_end = min(len(frames_bgr), impact_frame + track_only_frames - 1)
        tracked_boxes, last_helmet_box = track_helmet_window(track_start, track_end, helmet_box_xyxy)
        track_window_complete = (track_end - track_start + 1) >= track_only_frames

        motion_start = track_end + 1
        motion_end = min(len(frames_bgr), motion_start + motion_check_frames - 1)

        knocked_out_concussion_suspected = False
        all_overlays = {}

        if (not track_window_complete) or (motion_start > len(frames_bgr)):
            summary = {
                "head_med": np.nan,
                "body_med": np.nan,
                "helmet_med": np.nan,
                "head_max": np.nan,
                "body_max": np.nan,
                "helmet_max": np.nan,
                "enough_data": False,
            }
            purposeful = True
            motionless = False
            motion_window_complete = False
            stopped_early_for_motion = False
            stopped_early_for_visibility = True
            abandoned_for_missing_body = False
        else:
            (
                motions,
                helmet_motions,
                overlays,
                _,
                stopped_early_for_motion,
                stopped_early_for_visibility,
                abandoned_for_missing_body,
            ) = analyze_motion_window(
                motion_start, motion_end, last_helmet_box
            )
            all_overlays.update(overlays)

            purposeful, motionless, summary = evaluate_window(motions, helmet_motions)
            motion_window_complete = (motion_end - motion_start + 1) >= motion_check_frames

            if abandoned_for_missing_body:
                knocked_out_concussion_suspected = False
            elif stopped_early_for_visibility:
                knocked_out_concussion_suspected = False
            elif stopped_early_for_motion:
                knocked_out_concussion_suspected = False
            elif not motion_window_complete:
                knocked_out_concussion_suspected = False
            elif not summary["enough_data"]:
                knocked_out_concussion_suspected = False
            elif purposeful:
                knocked_out_concussion_suspected = False
            elif motionless:
                knocked_out_concussion_suspected = True
            else:
                knocked_out_concussion_suspected = False

        decision_frame = motion_end if knocked_out_concussion_suspected else -1

        event_rows.append({
            "event_id": event_id,
            "impact_frame": impact_frame,
            "helmet_left": int(row.left),
            "helmet_top": int(row.top),
            "helmet_width": int(row.width),
            "helmet_height": int(row.height),
            "det_score": float(row.det_score),
            "pred_cls": float(row.pred_cls),
            "impact_score": float(row.impact_score),
            "head_motion_after_wait": summary["head_med"],
            "body_motion_after_wait": summary["body_med"],
            "helmet_motion_after_wait": summary["helmet_med"],
            "head_max_after_wait": summary["head_max"],
            "body_max_after_wait": summary["body_max"],
            "helmet_max_after_wait": summary["helmet_max"],
            "purposeful_movement_after_wait": bool(purposeful),
            "motionless_after_wait": bool(motionless),
            "knocked_out_concussion_suspected": bool(knocked_out_concussion_suspected),
        })

        if knocked_out_concussion_suspected:
            if len(all_overlays) > 0:
                last_overlay_frame = max(all_overlays.keys())
                last_overlay = all_overlays[last_overlay_frame]
                final_helmet_box = last_overlay["helmet_box"]
                final_body_roi = last_overlay["body_roi"]
                final_head_roi = last_overlay.get("head_roi")
                final_torso_roi = last_overlay.get("torso_roi")
                final_head_prev_pts = last_overlay.get("head_prev_pts", [])
                final_head_curr_pts = last_overlay.get("head_curr_pts", [])
                final_body_prev_pts = last_overlay.get("body_prev_pts", [])
                final_body_curr_pts = last_overlay.get("body_curr_pts", [])
            else:
                final_helmet_box = last_helmet_box
                final_body_roi = None
                final_head_roi = None
                final_torso_roi = None
                final_head_prev_pts = []
                final_head_curr_pts = []
                final_body_prev_pts = []
                final_body_curr_pts = []

            display_end = min(len(frames_bgr), decision_frame + concussion_display_frames - 1)

            for fidx in range(decision_frame, display_end + 1):
                concussion_frame_overlays.setdefault(fidx, []).append({
                    "event_id": event_id,
                    "helmet_box": final_helmet_box,
                    "body_roi": final_body_roi,
                    "head_roi": final_head_roi,
                    "torso_roi": final_torso_roi,
                    "head_prev_pts": final_head_prev_pts,
                    "head_curr_pts": final_head_curr_pts,
                    "body_prev_pts": final_body_prev_pts,
                    "body_curr_pts": final_body_curr_pts,
                    "decision_frame": decision_frame,
                })

    events_df = pd.DataFrame(event_rows)
    events_df.to_csv(OPENCV_EVENTS_CSV, index=False)
    print(f"Saved OpenCV knockout/concussion summary: {OPENCV_EVENTS_CSV}")
    print(events_df[[
        "event_id",
        "impact_frame",
        "purposeful_movement_after_wait",
        "knocked_out_concussion_suspected"
    ]])

    print("Rendering OpenCV knockout/concussion video...")
    writer = cv2.VideoWriter(
        OPENCV_VIDEO_PATH,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (frame_w, frame_h)
    )

    flagged_events = set(
        events_df.loc[
            events_df["knocked_out_concussion_suspected"] == True,
            "event_id"
        ].tolist()
    )

    for frame_idx, frame in enumerate(tqdm(frames_bgr, desc="Render OpenCV video"), start=1):
        draw = frame.copy()

        for item in impact_frame_overlays.get(frame_idx, []):
            hx1, hy1, hx2, hy2 = item["helmet_box"]
            draw_helmet_highlight(
                draw,
                hx1, hy1, hx2 - hx1, hy2 - hy1,
                label="helmet impact",
                frame_idx=frame_idx,
                glow_color=HELMET_GLOW_COLOR,
                label_bg=HELMET_LABEL_BG,
            )

        for item in concussion_frame_overlays.get(frame_idx, []):
            if item["event_id"] not in flagged_events:
                continue
            if frame_idx < item["decision_frame"]:
                continue

            hx1, hy1, hx2, hy2 = item["helmet_box"]
            draw_helmet_highlight(
                draw,
                hx1, hy1, hx2 - hx1, hy2 - hy1,
                label="concussion impact",
                frame_idx=frame_idx,
                glow_color=CONCUSSION_GLOW_COLOR,
                label_bg=CONCUSSION_LABEL_BG,
            )

            if item["body_roi"] is not None:
                bx1, by1, bx2, by2 = item["body_roi"]
                cv2.rectangle(draw, (bx1, by1), (bx2, by2), BODY_BOX_COLOR, BODY_BOX_THICKNESS, cv2.LINE_AA)

            if item.get("head_roi") is not None:
                x1, y1, x2, y2 = item["head_roi"]
                cv2.rectangle(draw, (x1, y1), (x2, y2), HEAD_ROI_COLOR, 1, cv2.LINE_AA)

            if item.get("torso_roi") is not None:
                x1, y1, x2, y2 = item["torso_roi"]
                cv2.rectangle(draw, (x1, y1), (x2, y2), TORSO_ROI_COLOR, 1, cv2.LINE_AA)

            for p0, p1 in zip(item.get("head_prev_pts", []), item.get("head_curr_pts", [])):
                cv2.line(draw, p0, p1, HEAD_POINT_COLOR, POINT_LINE_THICKNESS, cv2.LINE_AA)
                cv2.circle(draw, p1, POINT_RADIUS, HEAD_POINT_COLOR, -1, cv2.LINE_AA)

            for p0, p1 in zip(item.get("body_prev_pts", []), item.get("body_curr_pts", [])):
                cv2.line(draw, p0, p1, BODY_POINT_COLOR, POINT_LINE_THICKNESS, cv2.LINE_AA)
                cv2.circle(draw, p1, POINT_RADIUS, BODY_POINT_COLOR, -1, cv2.LINE_AA)

            annotate_banner(draw, "KNOCKED OUT / CONCUSSION SUSPECTED")

        writer.write(draw)

    writer.release()

    print("\nDone with OpenCV pass.")
    print("OpenCV video:", OPENCV_VIDEO_PATH)
    print("Event CSV:", OPENCV_EVENTS_CSV)